In [13]:
import numpy as np
import ssqpy
from time import time, sleep

np.set_printoptions(suppress=True)

ssqpy.setSilentMode()

In [ ]:
from cloudpendulumclient.benchmark.controller import ControllerBase, MotorState

class SSQPYController(ControllerBase):
    def __init__(self):
        self.dt = 0.01
        MPC_H = 10
        
        V_WGT = np.array((7e-1, 2.4e-1))
        U_WGT = np.array((0.0, 4.2e-2))
        ELBOW_WGT = 200.0
        TIP_WGT = 120.0
        
        elbow_height = 0.1
        tip_height = 0.25
        
        torque_clip = 0.08
        
        vel_bounds = np.array((25, 25))
        self.torque_bounds = np.array((0.005, torque_clip,))
        
        self.model = ssqpy.model.Model(
            MPC_H,
            self.dt,
            urdf_path="acrobot.urdf",
            solver_mode=ssqpy.model.SolverMode.InverseDynamics,
        )
        
        self.nq = self.model.getnq()
        self.nv = self.model.getnv()
        self.nu = self.model.getnu()
        
        vel_cost = ssqpy.model.costs.SquaredJointVelocityCost(self.model, V_WGT)
        u_cost = ssqpy.model.costs.SquaredControlCost(self.model, U_WGT)
        elbow_cost = ssqpy.model.costs.FrameSquaredTranslationErrorCost(
            self.model, "link2", np.array((0.0, 0.0, elbow_height)), ELBOW_WGT
        )
        tip_cost = ssqpy.model.costs.FrameSquaredTranslationErrorCost(
            self.model, "tip", np.array((0.0, 0.0, tip_height)), TIP_WGT
        )
        
        for k in range(MPC_H):
            self.model.addCost(k, vel_cost)
            self.model.addCost(k, u_cost)
            self.model.addCost(k, elbow_cost)
            self.model.addCost(k, tip_cost)
        
        self.model.addCost(MPC_H, vel_cost)
        self.model.addCost(MPC_H, elbow_cost)
        self.model.addCost(MPC_H, tip_cost)
        
        self.model.finalize(
            custom_velocity_bounds=vel_bounds,
            custom_torque_bounds=self.torque_bounds,
        )
        
        ssqp_params = ssqpy.solvers.ssqpParams()
        ssqp_params.tolerance = 1e-2
        ssqp_params.qp_solver_type = ssqpy.solvers.QPSolverType.HPIPM
        
        hpipm_params = ssqpy.solvers.hpipmParams()
        hpipm_params.tol_comp = 1e-3
        hpipm_params.tol_stat = 1e-3
        hpipm_params.tol_eq = 1e-3
        hpipm_params.tol_ineq = 1e-3
        ssqp_params.hpipmParams = hpipm_params
        
        self.mpc = ssqpy.solvers.MPC(self.model, ssqp_params, sqp_iters=6, qp_iters=100)

    def get_control_output(self, state):
        mq = np.array([state[i].position for i in range(len(state))])
        mv = np.array([state[i].velocity for i in range(len(state))])

        try:
            u = self.mpc.step(np.hstack((mq, mv)))[1].stage(0)
        except RuntimeError:
            u = np.zeros(self.nv)

        tau = self.model.inverseDynamics(mq, mv, u)
        tau = np.clip(tau, -self.torque_bounds, self.torque_bounds)

        return tau

In [ ]:
from cloudpendulumclient.benchmark.control_loop import ControlLoop
from cloudpendulumclient.benchmark.evaluators.goal_height_evaluator import GoalHeightEvaluator

if __name__ == "__main__":
    controller = SSQPYController()
    evaluator = GoalHeightEvaluator()

    with open("../token.txt", "r") as f:
        token = f.readlines()[0].strip()
    
    control_loop = ControlLoop(
        user_token = token,
        experiment_time = 20.0,
        experiment_type = "DoublePendulum"
    )
    control_loop.start()

    while not control_loop.finished():
        state = control_loop.get_state()
        control_loop_time = control_loop.time()

        controller_output = controller.get_control_output(state)
        evaluator.evaluate(state, controller_output, control_loop_time)

        control_loop.step(controller_output)
    
    video_url, logs = control_loop.stop()

    score = evaluator.get_score()
    print("Score:", score)